# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [11]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

In [6]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining_proxy"] = (data["imp_h2"] < 0.8 * data["imp_h1"]).astype(int)
data = data.dropna(subset=["ctr_h1", "avg_position_h1"]).reset_index(drop=True)

def position_tier(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

data["position_tier"] = data["avg_position_h1"].apply(position_tier)
print(f"{len(data):,} rows | is_declining_proxy rate: {data['is_declining_proxy'].mean():.3f}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,475 rows | is_declining_proxy rate: 0.296


,client_hash_id,content_hash_id,imp_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,imp_h2,is_declining_proxy,position_tier
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,0.001438,6.327311,15,2350.0,1,4-10
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,0.000000,4.185913,15,208.0,0,4-10
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,0.000810,6.473735,15,1925.0,1,4-10
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,0.003279,7.259861,15,2504.0,0,4-10
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,0.0,0.000000,20.250000,9,28.0,0,21+


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:
print(data[["imp_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]].describe())
# TODO once run: note which columns have heavy tails (mean >> median is the tell) --
# imp_h1 and clicks_h1 are the likely candidates given how impressions data usually looks.

              imp_h1      clicks_h1         ctr_h1  avg_position_h1  \
count  120475.000000  120475.000000  120475.000000    120475.000000   
mean     1057.476564       3.186869       0.003010        16.225084   
std      2967.326193      14.800922       0.008678        16.906887   
min        10.000000       0.000000       0.000000         0.040763   
25%        55.000000       0.000000       0.000000         5.145136   
50%       216.000000       0.000000       0.000000         9.033372   
75%       856.000000       2.000000       0.002971        21.418724   
max    161575.000000    2395.000000       0.300000       127.620709   

       active_days_h1  
count   120475.000000  
mean        12.965769  
std          3.180422  
min          1.000000  
25%         12.000000  
50%         15.000000  
75%         15.000000  
max         15.000000  


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*



**Signal 1 — does `is_declining_proxy` rate fall as position gets worse, or improve?**
(if declines cluster in bad positions, position is a genuinely useful feature, not noise)

In [8]:
signal1 = data.groupby("position_tier")["is_declining_proxy"].agg(["mean", "count"]).reindex(["1-3", "4-10", "11-20", "21+"])
print(signal1)
# TODO once run: verdict -- CONFIRMED / OPPOSITE / MIXED / FALSE, one sentence why.

                   mean  count
position_tier                 
1-3            0.249953  10582
4-10           0.320753  53786
11-20          0.275206  23804
21+            0.286785  32303


**Signal 2 — does `active_days_h1` (how often a page shows up at all) relate to decline rate?**
(a page active fewer days in h1 might already be fading before h2 confirms it)

In [12]:
data["active_days_tier"] = pd.cut(data["active_days_h1"], bins=[-1, 5, 10, 15], labels=["1-5", "6-10", "11-15"])
signal2 = data.groupby("active_days_tier", observed=True)["is_declining_proxy"].agg(["mean", "count"])
print(signal2)
# TODO once run: verdict -- CONFIRMED / OPPOSITE / MIXED / FALSE, one sentence why.

                      mean  count
active_days_tier                 
1-5               0.203916   6282
6-10              0.321813  17013
11-15             0.297963  97180


**Signal 3 — does `ctr_h1` (click-through rate in the first half of March) relate to decline rate?**
(a page already under-converting its impressions might be more likely to keep losing them)

In [13]:
data["ctr_tier"] = pd.qcut(data["ctr_h1"], q=4, duplicates="drop")
signal3 = data.groupby("ctr_tier", observed=True)["is_declining_proxy"].agg(["mean", "count"])
print(signal3)
# TODO once run: verdict -- CONFIRMED / OPPOSITE / MIXED / FALSE, one sentence why.

                       mean  count
ctr_tier                          
(-0.001, 0.00297]  0.325302  90356
(0.00297, 0.3]     0.209801  30119


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [14]:
quick_win_check = data.groupby("position_tier")["imp_h1"].agg(["mean", "median", "count"]).reindex(["1-3", "4-10", "11-20", "21+"])
print(quick_win_check)
# TODO once run: does the median for "11-20" beat "21+"? That's the quick-win assumption --
# same mean-vs-median caveat as the Week 4 baseline notebook: check both, trust the median if impressions are skewed.

                      mean  median  count
position_tier                            
1-3            1870.525137   740.0  10582
4-10           1145.259826   295.0  53786
11-20           632.347379   188.0  23804
21+             958.246912    92.0  32303


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

CTR is the one clean, trustworthy signal here — low-CTR pages decline at roughly 1.5x the rate of high-CTR ones, and it's monotonic across the whole range. Position on its own is weaker than expected — only the very top tier is meaningfully protected, and the middle tiers don't follow a clean pattern, so it shouldn't be trusted alone. Active-days behaved backwards from the hypothesis, which is a useful negative result: a page being quiet isn't the same as a page declining — it may just have less traffic left to lose. The quick-win rule's volume assumption checks out, so that part of the baseline is standing on solid ground.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.